# HAR Walk-Forward Benchmark Report

This report evaluates HAR benchmarking with **walk-forward training** using both:
- `expanding` window
- `rolling` window

It compares model types (`ols`, `lasso`, `bsr`) and feature-set ladders, and reports both **train** and **test** metrics.

In [ ]:
%matplotlib inline

import json
import logging
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.benchmark import WheatHARBenchmarkConfig, benchmark_results_to_frame, run_wheat_har_benchmark
from src.util.path import PROJECT_ROOT

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(name)s | %(message)s")
pd.set_option("display.max_columns", 300)

## Load Config and Run Benchmark

In [ ]:
CONFIG_PATH = PROJECT_ROOT / "config" / "har.json"

if CONFIG_PATH.exists():
    with CONFIG_PATH.open("r", encoding="utf-8") as f:
        cfg_raw = json.load(f)

    cfg = WheatHARBenchmarkConfig(
        csv_path=cfg_raw.get("csv_path", "data/ag/v4.csv"),
        target_col=cfg_raw.get("target_col", "wheat_weekly_rv"),
        core_columns=cfg_raw.get("core_columns"),
        target_horizon=int(cfg_raw.get("target_horizon", 1)),
    )
    print(f"Loaded config from: {CONFIG_PATH}")
else:
    cfg = WheatHARBenchmarkConfig()
    cfg_raw = {}
    print("Config not found; using defaults")

results = run_wheat_har_benchmark(config=cfg)
summary = benchmark_results_to_frame(results)

output_csv = Path("/data/benchmark/har.csv")
output_csv.parent.mkdir(parents=True, exist_ok=True)
summary.to_csv(output_csv, index=False)
print(f"Saved summary to: {output_csv}")

summary.head(20)

## Config Used

In [ ]:
pd.json_normalize(cfg_raw, sep=".").T if cfg_raw else pd.DataFrame({"config": ["default"]})

## Workflow Summary

For each run config and feature set:
1. Build design matrix from configured core + extra features.
2. Generate walk-forward windows (`expanding` or `rolling`).
3. For each window: select features on train window, fit HAR, predict train/test window.
4. Aggregate window predictions, then compute train and test metrics.

## Metrics Table (Train + Test)

In [ ]:
metric_cols = [
    "model_type",
    "feature_set",
    "window_type",
    "n_windows",
    "n_selected",
    "train_mse",
    "train_mae",
    "train_qlike",
    "train_r2",
    "train_r2log",
    "test_mse",
    "test_mae",
    "test_qlike",
    "test_r2",
    "test_r2log",
]

report = summary[metric_cols + [c for c in summary.columns if c not in metric_cols]].copy()
report = report.sort_values(["test_mse", "test_mae"]).reset_index(drop=True)
report.head(25)

## Hyperparameter Table

In [ ]:
hyper_cols = [
    "model_type",
    "feature_set",
    "window_type",
    "initial_train_size",
    "window_test_size",
    "window_step",
    "rolling_window_size",
    "selection_method",
    "target_col_raw",
    "target_horizon",
    "core_columns",
    "extra_feature_cols",
    "model_add_constant",
    "model_standardize_features",
    "model_target_transform",
    "model_prediction_floor",
    "model_log_transform_rv_features",
    "model_feature_floor",
    "model_max_selected_features",
    "model_min_train_feature_ratio",
    "lasso_best_alpha",
    "bsr_alpha",
    "bsr_window_type",
    "bsr_window_size",
    "bsr_step",
]

available_hyper_cols = [c for c in hyper_cols if c in report.columns]
report[available_hyper_cols].head(30)

## Compare Expanding vs Rolling (Test MSE)

In [ ]:
pivot_mse = report.pivot_table(
    index=["feature_set", "model_type"],
    columns="window_type",
    values="test_mse",
    aggfunc="mean",
)

pivot_mse

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
for wtype, group in report.groupby("window_type"):
    top = group.sort_values("test_mse").head(10).copy()
    ax.plot(range(len(top)), top["test_mse"], marker="o", label=wtype)

ax.set_title("Top-10 Test MSE by Window Type")
ax.set_xlabel("Rank")
ax.set_ylabel("Test MSE")
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

## Train vs Test Generalization

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
for wtype, group in report.groupby("window_type"):
    ax.scatter(group["train_mse"], group["test_mse"], label=wtype, s=60)

mx = max(report["train_mse"].max(), report["test_mse"].max())
ax.plot([0, mx], [0, mx], linestyle="--")
ax.set_xlabel("Train MSE")
ax.set_ylabel("Test MSE")
ax.set_title("Generalization: Train vs Test")
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

## Best Run: Metrics and Coefficients

In [ ]:
best = report.sort_values(["test_mse", "test_mae"]).iloc[0]
best_model_type = best["model_type"]
best_feature_set = best["feature_set"]

best_result = results[best_model_type][best_feature_set]

best_metrics = pd.DataFrame([
    {"split": "train", **best_result.metrics["train"]},
    {"split": "test", **best_result.metrics["test"]},
]).set_index("split")

best_coefs = best_result.coefficients.to_frame("coefficient")
best_coefs["abs_coefficient"] = best_coefs["coefficient"].abs()
best_coefs = best_coefs.sort_values("abs_coefficient", ascending=False)

print(f"Best run: {best_model_type} | {best_feature_set}")
display(best_metrics)
display(best_coefs)

## Best Run Prediction Plot (Test)

In [ ]:
plot_df = pd.DataFrame({
    "y_true_test": best_result.y_true_test,
    "y_pred_test": best_result.y_pred_test,
}).sort_index()

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(plot_df.index, plot_df["y_true_test"], label="Actual")
ax.plot(plot_df.index, plot_df["y_pred_test"], label="Predicted")
ax.set_title("Best Run Test Predictions")
ax.set_ylabel("RV")
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()